# 01 — Forensic Exploratory Data Analysis & Data Quality Audit

**Purpose:** Audit raw competition files, detect hardware sentinel spikes, verify exact timestamp alignment, uncover missing date holes, inspect post-Diwali dispersion physics, and triage `AdditionalHistory.zip` **strictly before any machine learning modeling begins**.

**Guardrails:**
- Zero ML modeling code in this notebook.
- `train_raw` and `test_raw` are read-only and never overwritten in place.
- Outputs verified clean `.parquet` checkpoints into `data/processed/` for `02_modeling_pipeline.ipynb`.

In [ ]:
# Block 0: Local Environment Preflight & Path Discovery
import sys
from pathlib import Path

# Ensure repository root is on sys.path so `src.*` imports cleanly from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import resolve_raw_data_dir, run_preflight_check
from src.cleaning import (
    align_history_2024,
    apply_stage1_hardware_scrub,
    apply_stage2_spatial_scrub,
    load_historical_archive,
    load_raw_core_datasets,
    standardize_historical_df,
)

env_info = run_preflight_check()
RAW_DIR = resolve_raw_data_dir()

## 1. Schema, Duplicate Columns & Test ID Alignment Audit
Verify row counts, timezone alignment (`timestamp_ist` matches `target_end_ist`), and whether `Test.csv` is chronologically unsorted relative to `SampleSubmission.csv`.

In [ ]:
# Load raw core datasets into read-only DataFrames
train_raw, test_raw, sample_sub_raw, exact_station_map = load_raw_core_datasets(RAW_DIR)

print("\n[EDA Check 1A] Test.csv Scoring Block Breakdown:")
print(test_raw["scoring_block"].value_counts().to_string())

is_test_sorted = test_raw["target_end_ist"].is_monotonic_increasing
id_matches_sample = bool((test_raw["ID"].values == sample_sub_raw["ID"].values).all())
print(f"\n[EDA Check 1B] Test.csv chronologically sorted? {is_test_sorted} (First target_end_ist: {test_raw['target_end_ist'].iloc[0]})")
print(f"[EDA Check 1C] Test.csv ID order matches SampleSubmission.csv exactly? {id_matches_sample}")
print(f"[EDA Check 1D] Exact Target Station Map (4 Stations): {exact_station_map}")

## 2. Target Quantiles & Two-Stage Hardware / Spatial Glitch Scrubbing Audit
Inspect extreme upper spikes (`99999.0` CPCB BAM error codes / `flag_high_1000 == 1`) and post-reboot zero dropouts (`< 2.0`), then audit Stage 1 and Stage 2 scrubbing using the glass-box `trace` dictionary.

In [ ]:
# Inspect raw target quantiles before cleaning
q_levels = [0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
raw_quantiles = train_raw["pm25_raw"].quantile(q_levels)
print("[EDA Check 2A] Raw pm25 Quantiles (Look for 0.0 dropouts & 99999 sentinel spikes):")
print(raw_quantiles.to_string())

# Run glass-box scrubbing with trace={} so flagged rows can be inspected in VS Code Data Wrangler
eda_trace = {}
raw_aux = load_historical_archive(RAW_DIR, ["Other_Mumbai_Stations_2025_2026"])
aux_clean = standardize_historical_df(raw_aux, "Other_Mumbai_Stations_2025_2026") if raw_aux is not None else None

train_hw_scrubbed = apply_stage1_hardware_scrub(train_raw, trace=eda_trace)
train_clean = apply_stage2_spatial_scrub(train_hw_scrubbed, aux_clean=aux_clean, safe_aux=True, trace=eda_trace)

print("\n[EDA Check 2B] Sample Flagged Stage-2 Spatial Anomaly Rows (Inspectable via eda_trace):")
print(eda_trace["stage2_flagged_spatial_rows"][["timestamp_ist", "location_id", "pm25_raw"]].head(10).to_string(index=False))

## 3. Temporal Continuity & Withheld Date-Hole Audit
Detect multi-day missing holes inside `Train.csv` (such as the withheld October 24-28, 2026 window) that would break naive row-index `.shift()` lags.

In [ ]:
# Check daily observation counts in late October & early November 2026
late_oct_nov = train_clean[train_clean["timestamp_ist"] >= pd.Timestamp("2026-10-18 00:00:00+05:30")].copy()
late_oct_nov["date_ist"] = late_oct_nov["timestamp_ist"].dt.date

daily_audit = late_oct_nov.groupby("date_ist")["pm25"].agg(["count", "mean", "median", "max"])
print("[EDA Check 3A] Late Oct - Nov 2026 Daily Observation Counts (Verify Oct 24-28 hole & Nov 8-10 Diwali spike):")
print(daily_audit.to_string())

# Confirm all Train.csv timestamps are strictly on top-of-hour (:00:00)
non_zero_min = int((train_clean["timestamp_ist"].dt.minute != 0).sum())
print(f"\n[EDA Check 3B] Non-top-of-hour timestamps in train_clean: {non_zero_min} (Expected: 0)")

## 4. Historical Diwali 2025 Post-Peak Dispersion & Early November 2026 Station Regimes
Compare Diwali 2025 (peak Oct 20-21 -> Day +3/+4 Oct 23-24) against Pre-Diwali vs. Post-Diwali 2026 station telemetry to calibrate Private LB (`Nov 11-12`) domain expectations.

In [ ]:
# Diagnostic 2A: Historical Diwali 2025 decay trajectory
d25 = train_clean[
    (train_clean["timestamp_ist"] >= pd.Timestamp("2025-10-19 00:00:00+05:30"))
    & (train_clean["timestamp_ist"] <= pd.Timestamp("2025-10-25 23:00:00+05:30"))
].copy()
d25["date_ist"] = d25["timestamp_ist"].dt.date
print("[EDA Check 4A] Historical Diwali 2025 Trajectory (Peak Oct 20-21 -> Sea-Breeze Dispersion Oct 23-25):")
print(d25.groupby("date_ist")["pm25"].agg(["count", "mean", "median", "max"]).to_string())

# Diagnostic 1D: Per-station breakdown Pre-Diwali (Nov 4-7) vs Diwali/Post (Nov 8-10)
nov_slice = train_raw[train_raw["timestamp_ist"] >= pd.Timestamp("2026-11-04 00:00:00+05:30")].copy()
nov_slice = nov_slice.merge(
    train_clean[["timestamp_ist", "location_id", "pm25"]].rename(columns={"pm25": "pm25_clean"}),
    on=["timestamp_ist", "location_id"],
    how="left",
)
nov_slice["regime_period"] = np.where(
    nov_slice["timestamp_ist"] < pd.Timestamp("2026-11-08 00:00:00+05:30"),
    "1_PreDiwali_Nov04_07",
    "2_Diwali_Post_Nov08_10",
)
st_regime = nov_slice.groupby(["location_id", "station_name", "regime_period"]).agg(
    n_clean=("pm25_clean", "count"),
    clean_mean=("pm25_clean", "mean"),
    clean_med=("pm25_clean", "median"),
    raw_mean=("pm25_raw", "mean"),
)
print("\n[EDA Check 4B] Per-Station Telemetry Breakdown (Pre-Diwali vs Nov 8-10):")
print(st_regime.to_string())

## 5. Additional History Triage & Frozen Parquet Checkpoint Export
Audit `History_Mumbai_2024` (sub-hourly `:30:00` ceiling, Trap 1 exact station matching, Trap 2 verified site filtering, and per-station Median-IQR alignment) and save frozen checkpoints to `data/processed/`.

In [ ]:
from src.config import PIPELINE_CONFIG, PROCESSED_DATA_DIR

raw_hist_2024 = load_historical_archive(RAW_DIR, ["train_mumbai_2024_auxiliary", "History_Mumbai_2024"])
hist_2024_aligned = None

if raw_hist_2024 is not None:
    hist_2024_aligned = align_history_2024(
        raw_hist_2024=raw_hist_2024,
        train_clean=train_clean,
        exact_station_map=exact_station_map,
        months=PIPELINE_CONFIG["hist_2024_months"],
        align_method=PIPELINE_CONFIG["align_method_2024"],
        trace=eda_trace,
    )
    cal_tbl = eda_trace["hist_2024_calibration_table"]
    print("\n[EDA Check 5A] 2024 Calibration Alignment Audit (Unaligned vs Aligned Median by Station):")
    print(
        cal_tbl.groupby("location_id").agg(
            n_rows=("pm25", "count"),
            unaligned_med=("pm25_unaligned", "median"),
            aligned_med=("pm25", "median"),
        ).to_string()
    )
else:
    print("[EDA Check 5A] History_Mumbai_2024 not found in data/raw/ (optional).")

# Export frozen, clean Parquet checkpoints for 02_modeling_pipeline.ipynb
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_clean.to_parquet(PROCESSED_DATA_DIR / "train_clean.parquet", index=False)
if hist_2024_aligned is not None:
    hist_2024_aligned.to_parquet(PROCESSED_DATA_DIR / "hist_2024_aligned.parquet", index=False)

print(f"\n[EDA Check 5B] Saved frozen Parquet checkpoints to {PROCESSED_DATA_DIR}. Phase 1 EDA complete!")